# Hack1Robo · Décodage par token

Exécute **Préparation** dans tous les cas. La section **API hébergée** fonctionne sans GPU avec une clé. La section **Décodage exact** utilise le GPU de Colab et fonctionne sans clé. Pour activer le GPU, sélectionne **Exécution → Modifier le type d’exécution → GPU**.

Les deux sections peuvent être lancées indépendamment.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from importlib import reload
import json

url = "https://raw.githubusercontent.com/antoineach/Hack1Robo2026/main/hack1robo.py"
if not Path("hack1robo.py").exists():
    urlretrieve(url, "hack1robo.py")

import hack1robo as hack
hack = reload(hack)

def secret(name):
    from google.colab import userdata
    return userdata.get(name)

default_prompt = hack.PROMPTS[0]

## API hébergée · GPU facultatif

Avec **aucune clé**, le catalogue public indique quels modèles OpenRouter gratuits déclarent des probabilités. Pour la démo ou la sonde, ajoute le secret `OPENROUTER_API_KEY` ou `GEMINI_API_KEY` dans la barre latérale de Colab et donne au notebook l’accès au secret.

Les trois blocs utilisent des températures différentes. Le texte précédent est envoyé dans la consigne de chaque appel. La sonde peut répondre par une erreur si le fournisseur ne propose pas les probabilités. Les options sont désactivées au départ pour préserver ton quota.

In [ ]:
print("Modèles OpenRouter gratuits déclarant top_logprobs :")
print(hack.openrouter_free_catalog())

In [ ]:
RUN_BLOCKS = False
RUN_PROBE = False
PROVIDER = "openrouter"  # ou "gemini"
PROBE_OPENROUTER_MODEL = "liquid/lfm-3-1.7b:free"

if RUN_BLOCKS or RUN_PROBE:
    key = secret("OPENROUTER_API_KEY" if PROVIDER == "openrouter" else "GEMINI_API_KEY")
    if RUN_BLOCKS:
        hosted = hack.HostedModel(PROVIDER, key)
        print(json.dumps(hack.hosted_demo(hosted, default_prompt), ensure_ascii=False, indent=2))
    if RUN_PROBE:
        model = PROBE_OPENROUTER_MODEL if PROVIDER == "openrouter" else hack.GEMINI_GEMMA
        probe = hack.HostedModel(PROVIDER, key, model)
        try:
            print(json.dumps(probe.probe(), ensure_ascii=False, indent=2))
        except RuntimeError as error:
            print(f"Sonde refusée : {error}")

## Décodage exact · GPU Colab requis

Le modèle utilisé est **Gemma 4 E2B Instruct**. Compte plusieurs Go de téléchargement dans Colab au premier chargement. Ton ordinateur affiche seulement le notebook. Le processeur de logits applique la distribution avant chaque tirage, et le graphique montre la température prescrite et l’entropie réellement observée.

In [ ]:
%pip -q install "transformers==5.5.4" "accelerate>=1.0,<2" pillow

In [ ]:
decoder = hack.GemmaDecoder()

In [ ]:
from dataclasses import replace
import matplotlib.pyplot as plt

profile = replace(hack.DEFAULT, mode="softmax")  # "mixture" ou "entmax"
if profile.mode == "entmax":
    profile = replace(profile, strength=1.0)
example = decoder.generate(default_prompt, profile, seed=17, min_tokens=112)
print(example["text"])

ts = [row["t"] for row in example["tokens"]]
fig, (temperature, entropy) = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
temperature.plot(ts, [row["temperature"] for row in example["tokens"]], color="tab:orange")
temperature.axhline(profile.nominal, linestyle=":", color="gray")
temperature.set_ylabel("Température")
entropy.plot(ts, [row["entropy"] for row in example["tokens"]], color="tab:blue")
entropy.set(xlabel="Position du token dans la réponse (base 0)", ylabel="Entropie (nats)")
for ax in (temperature, entropy):
    ax.axvline(profile.peak_at, alpha=.35, color="gray")
    ax.axvline(profile.crash_at, alpha=.35, color="gray")
fig.tight_layout()
plt.show()

### Jugement à l’aveugle et recherche avec trois agents

Le premier lot est volontairement court. Il compare les deux contrôles, trois distributions et un profil par agent. La validation utilise un prompt distinct. Augmente les listes de prompts et de graines pour un résultat plus stable. Une expérience plus grande peut consommer davantage de quota API si tu choisis un juge externe.

In [ ]:
import itertools
from IPython.display import display
import pandas as pd

JUDGE = "local"  # "local", "gemini" ou "openrouter"
AGENTS = "local"  # "local" ou "judge" pour utiliser le même fournisseur que le juge
AGENT_ROUNDS = 1
TRAIN = hack.PROMPTS[:1]
VALIDATION = hack.HOLDOUT[:1]
SEEDS = (17,)  # ex. (17, 41)

local_seeds = itertools.count(4000)

def local_llm(prompt, *, temperature, max_tokens):
    profile = replace(hack.CONTROL, nominal=temperature, peak=temperature, crash=temperature)
    return decoder.generate(prompt, profile, seed=next(local_seeds), max_tokens=max_tokens)["text"]

if JUDGE == "local":
    judge = local_llm
elif JUDGE == "gemini":
    judge = hack.HostedModel("gemini", secret("GEMINI_API_KEY"), "gemini-flash-lite-latest").text
elif JUDGE == "openrouter":
    judge = hack.HostedModel("openrouter", secret("OPENROUTER_API_KEY")).text
else:
    raise ValueError("Juge inconnu.")

agent_llm = local_llm if AGENTS == "local" else judge
agents = dict.fromkeys(hack.ROLES, agent_llm)
report = hack.experiment(decoder, judge, agents=agents, rounds=AGENT_ROUNDS,
                         prompts=TRAIN, holdout=VALIDATION, seeds=SEEDS)
print("Variante retenue sur le lot d’exploration :", report["winner_exploration"])
display(pd.DataFrame(report["classement"]))
display(pd.DataFrame(report["validation_classement"]))
if report["agents_fallbacks"]:
    print("Propositions d’agents remplacées par un tirage aléatoire :", report["agents_fallbacks"])

for row in report["validation"]:
    print(f"\n{row['candidate']} · graine {row['seed']}\n{row['text']}")

In [ ]:
from google.colab import files

path = Path("hack1robo-resultats.json")
path.write_text(json.dumps({"exemple": example, **report}, ensure_ascii=False, indent=2), encoding="utf-8")
files.download(str(path))